# FragPipe Multi-Plex Validation (Runs 1–5)

Loads the first N completed plex results and produces:
- **Per-plex histograms**: unweighted ratio + best-charge-state-per-peptide ratio
- **Cross-run box plot**: enrichment distribution with each plex as one box
- **Intensity trend**: ratio vs precursor intensity quintile, one panel per plex

In [ ]:
import glob
import os
import re
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from collections import defaultdict

# ── CONFIG ────────────────────────────────────────────────────────────────────
PLEX_LIST_PATH = "/scratch/leduc.an/AAS_Evo/MS_SEARCH/plex_list.txt"
RESULTS_BASE   = "/scratch/leduc.an/AAS_Evo/MS_SEARCH/results"
REPO_DIR       = "/home/leduc.an/AAS_Evo_project/AAS_Evo"
TMT_MAP        = f"{REPO_DIR}/metadata/PDC_meta/pdc_file_tmt_map.tsv"
GDC_META       = f"{REPO_DIR}/metadata/GDC_meta/gdc_meta_matched.tsv"
FASTA_DIR      = "/scratch/leduc.an/AAS_Evo/FASTA/per_sample"
MISSENSE       = "/scratch/leduc.an/AAS_Evo/VEP/all_missense_mutations.tsv"
REF_FASTA      = "/scratch/leduc.an/AAS_Evo/SEQ_FILES/uniprot_human_canonical.fasta"
OUT_DIR        = "/scratch/leduc.an/AAS_Evo/MS_SEARCH/validation_multi"

N_PLEXES = 5   # first N plexes from plex_list.txt

os.makedirs(OUT_DIR, exist_ok=True)

with open(PLEX_LIST_PATH) as f:
    plex_ids = [l.strip() for l in f if l.strip()][:N_PLEXES]

print(f"Plexes to analyze ({len(plex_ids)}):")
for i, p in enumerate(plex_ids, 1):
    print(f"  {i}. {p}")

CHANNEL_ORDER = ["126","127N","127C","128N","128C","129N","129C","130N","130C","131N","131C"]
TMT_CHANNEL_MAP = {
    "tmt_126":"126",  "tmt_127n":"127N", "tmt_127c":"127C",
    "tmt_128n":"128N","tmt_128c":"128C", "tmt_129n":"129N",
    "tmt_129c":"129C","tmt_130n":"130N", "tmt_130c":"130C",
    "tmt_131":"131N", "tmt_131c":"131C",
}

In [ ]:
# ── LOAD SHARED RESOURCES (once, reused across all plexes) ───────────────────
AA3TO1 = {
    "Ala":"A","Arg":"R","Asn":"N","Asp":"D","Cys":"C","Gln":"Q","Glu":"E",
    "Gly":"G","His":"H","Ile":"I","Leu":"L","Lys":"K","Met":"M","Phe":"F",
    "Pro":"P","Ser":"S","Thr":"T","Trp":"W","Tyr":"Y","Val":"V",
}

def parse_hgvsp_to_swap(hgvsp):
    m = re.search(r'p\.([A-Z][a-z]{2})(\d+)([A-Z][a-z]{2})', str(hgvsp))
    if m:
        ref = AA3TO1.get(m.group(1))
        alt = AA3TO1.get(m.group(3))
        if ref and alt:
            return f"{ref}{m.group(2)}{alt}"
    return None

# gene symbol → UniProt accession (from GN= field in reference FASTA)
gene_to_acc = {}
with open(REF_FASTA) as f:
    for line in f:
        if line.startswith(">"):
            m = re.search(r'GN=(\S+)', line)
            if m:
                gene_to_acc[m.group(1)] = line.split("|")[1]
print(f"Gene → accession entries: {len(gene_to_acc):,}")

tmt = pd.read_csv(TMT_MAP, sep="\t")
gdc = pd.read_csv(GDC_META, sep="\t")
if "file_id" in gdc.columns and "gdc_file_id" not in gdc.columns:
    gdc = gdc.rename(columns={"file_id": "gdc_file_id"})
print(f"TMT map rows: {len(tmt):,}   GDC meta rows: {len(gdc):,}")

print("Loading missense table (shared filter base)...")
missense = pd.read_csv(MISSENSE, sep="\t", low_memory=False)
print(f"Total missense rows: {len(missense):,}")

In [ ]:
# ── PROCESSING HELPERS ────────────────────────────────────────────────────────
PRECURSOR_CANDIDATES = ["Intensity", "Precursor Intensity", "MS1 Intensity",
                        "PrecursorIntensity", "precursor_intensity"]

def find_ri_cols(df):
    found = {}
    for ch in CHANNEL_ORDER:
        if ch in df.columns:
            found[ch] = ch; continue
        cands = [c for c in df.columns if c.startswith("Intensity") and c.endswith(f"_{ch}")]
        if cands:
            found[ch] = cands[0]; continue
        cands = [c for c in df.columns if c.startswith(ch) and "intensity" in c.lower()]
        if cands:
            found[ch] = cands[0]
    return found

def parse_mapped_proteins(mapped_str):
    """'sp|P01889-S35A-B4B8|HLA-B-mut, ...' → {('P01889','S35A'), ...}"""
    if pd.isna(mapped_str):
        return set()
    pairs = set()
    for entry in str(mapped_str).split(","):
        parts = entry.strip().split("|")
        if len(parts) >= 2:
            pid_parts = parts[1].split("-")
            if len(pid_parts) >= 3:
                pairs.add((pid_parts[0], pid_parts[1]))
    return pairs

def parse_protein_id(protein_id):
    """'P11047-L888P-F262' → {('P11047','L888P')}."""
    if pd.isna(protein_id):
        return set()
    parts = str(protein_id).split("-")
    if len(parts) >= 3:
        return {(parts[0], parts[1])}
    return set()


def process_plex(plex_id):
    """
    Full ratio computation pipeline for one TMT plex.

    Returns a DataFrame with columns:
        Peptide, ratio, ratio_vaf, precursor_intensity, n_have_ch, n_not_have_ch,
        n_fasta_channels
    or None if psm.tsv is not found / no mutant PSMs.

    Same-patient exclusion: if a patient has channels on both sides of a mutation
    (e.g. their tumor is a "have" channel and their normal is a "not-have" channel),
    all channels from that patient are excluded from the not-have set to avoid
    comparing a patient against themselves.
    """
    results_dir = os.path.join(RESULTS_BASE, plex_id)
    psm_matches = sorted(glob.glob(os.path.join(results_dir, "*_1/psm.tsv")))
    if not psm_matches:
        print(f"  [{plex_id[:40]}] No psm.tsv — skipping")
        return None

    psm = pd.read_csv(psm_matches[0], sep="\t", low_memory=False)
    ri_col_map = find_ri_cols(psm)
    ri_cols    = list(ri_col_map.values())

    mut_all = psm[psm["Entry Name"].str.endswith("-mut", na=False)].copy()
    mut_psm = mut_all[(mut_all[ri_cols].fillna(0) > 0).any(axis=1)].copy() \
              if ri_cols else mut_all.copy()

    precursor_col = next((c for c in PRECURSOR_CANDIDATES if c in mut_psm.columns), None)

    # ── Channel → UUID → per-sample FASTA → mutation map ─────────────────────
    plex_tmt = (tmt[tmt["run_metadata_id"] == plex_id]
                [["tmt_channel","case_submitter_id","sample_type"]].drop_duplicates())
    plex_tmt = plex_tmt[~plex_tmt["case_submitter_id"].str.lower()
                        .isin(["ref","reference","pooled","pool","nan"])]
    plex_tmt["channel"] = plex_tmt["tmt_channel"].map(TMT_CHANNEL_MAP)

    plex_meta = plex_tmt.merge(
        gdc[["gdc_file_id","case_submitter_id","sample_type"]],
        on=["case_submitter_id","sample_type"], how="left")

    # ── Same-patient exclusion: channel → case mapping ────────────────────────
    channel_to_case = {}
    for _, row in plex_meta.iterrows():
        if pd.notna(row.get("channel")) and pd.notna(row.get("case_submitter_id")):
            channel_to_case[row["channel"]] = row["case_submitter_id"]

    mutation_to_channels = defaultdict(set)
    channels_with_fastas = set()
    for _, row in plex_meta.iterrows():
        uuid, channel = row["gdc_file_id"], row["channel"]
        if pd.isna(uuid) or pd.isna(channel):
            continue
        fasta_path = os.path.join(FASTA_DIR, f"{uuid}_mutant.fasta")
        if not os.path.exists(fasta_path):
            continue
        channels_with_fastas.add(channel)
        with open(fasta_path) as f:
            for line in f:
                if not line.startswith(">"): continue
                parts = line[1:].strip().split("|")
                if len(parts) >= 4 and parts[0] == "mut":
                    mutation_to_channels[(parts[1], parts[3])].add(channel)

    # ── VAF lookup ─────────────────────────────────────────────────────────────
    uuid_to_channel = (plex_meta.dropna(subset=["gdc_file_id","channel"])
                                .set_index("gdc_file_id")["channel"].to_dict())
    plex_missense = missense[missense["sample_id"].isin(uuid_to_channel)]

    mutation_channel_vaf = {}
    for _, vrow in plex_missense.iterrows():
        vaf     = vrow.get("VAF", np.nan)
        channel = uuid_to_channel.get(vrow["sample_id"])
        if pd.isna(vaf) or not channel: continue
        acc  = gene_to_acc.get(str(vrow.get("SYMBOL", "")))
        swap = parse_hgvsp_to_swap(str(vrow.get("HGVSp", "")))
        if acc and swap:
            mutation_channel_vaf[(acc, swap, channel)] = float(vaf)

    # ── Compute per-PSM ratios ─────────────────────────────────────────────────
    rows = []
    n_same_patient_excluded = 0
    for _, row in mut_psm.iterrows():
        mutations = parse_mapped_proteins(row.get("Mapped Proteins", float("nan")))
        if not mutations:
            mutations = parse_protein_id(row.get("Protein ID", float("nan")))
        if not mutations: continue

        have_channels = set()
        for mk in mutations:
            have_channels |= mutation_to_channels.get(mk, set())
        have_channels &= channels_with_fastas

        # Exclude all channels from the same patient as any have-channel
        have_cases = {channel_to_case.get(ch) for ch in have_channels} - {None}
        not_have = channels_with_fastas - have_channels
        same_patient_excluded = {ch for ch in not_have
                                  if channel_to_case.get(ch) in have_cases}
        not_have -= same_patient_excluded
        n_same_patient_excluded += len(same_patient_excluded)

        if not have_channels or not not_have: continue

        have_ri = [(ch, row[ri_col_map[ch]]) for ch in have_channels
                   if ch in ri_col_map and pd.notna(row[ri_col_map[ch]])]
        not_ri  = [row[ri_col_map[ch]] for ch in not_have
                   if ch in ri_col_map and pd.notna(row[ri_col_map[ch]])]
        if not have_ri or not not_ri: continue

        mean_not  = np.mean(not_ri)
        mean_have = np.mean([r for _, r in have_ri])
        ratio     = mean_have / mean_not if mean_not > 0 else np.nan

        weighted_pairs = []
        for ch, ri_val in have_ri:
            for acc, sw in mutations:
                v = mutation_channel_vaf.get((acc, sw, ch))
                if v is not None:
                    weighted_pairs.append((ri_val, v)); break
        if weighted_pairs:
            ris_v, wts_v = zip(*weighted_pairs)
            ratio_vaf = np.average(ris_v, weights=wts_v) / mean_not if mean_not > 0 else np.nan
        else:
            ratio_vaf = np.nan

        prec = float(row[precursor_col]) \
            if (precursor_col and pd.notna(row.get(precursor_col))
                and row.get(precursor_col, 0) > 0) else np.nan

        rows.append({
            "Peptide":             row.get("Peptide", ""),
            "ratio":               ratio,
            "ratio_vaf":           ratio_vaf,
            "precursor_intensity": prec,
            "n_have_ch":           len(have_channels),
            "n_not_have_ch":       len(not_have),
            "n_fasta_channels":    len(channels_with_fastas),
        })

    ratios = pd.DataFrame(rows).dropna(subset=["ratio"])
    n_uniq = ratios.drop_duplicates(subset=["Peptide"])["ratio"].notna().sum()
    print(f"  [{plex_id[:40]}]  "
          f"mut PSMs: {len(mut_psm):,}  →  ratios: {len(ratios):,}  "
          f"(uniq pep: {n_uniq:,},  FASTA ch: {len(channels_with_fastas)},  "
          f"same-patient excluded: {n_same_patient_excluded:,})")
    return ratios


# ── RUN ALL PLEXES ─────────────────────────────────────────────────────────────
print("Processing plexes...\n")
all_ratios = {}   # plex_id → ratios DataFrame
for plex_id in plex_ids:
    df = process_plex(plex_id)
    if df is not None and len(df) > 0:
        all_ratios[plex_id] = df

print(f"\nDone: {len(all_ratios)} / {len(plex_ids)} plexes with ratio data")

In [ ]:
# ── PLOT 1: Per-plex histograms — 2 rows × N_PLEXES columns ──────────────────
# Row 0: all PSMs unweighted
# Row 1: best charge state per peptide (deduplicated by highest precursor intensity)

n = len(all_ratios)
plex_list_ordered = list(all_ratios.keys())

# Short label: pull tissue/study segment from plex ID
def short_label(plex_id):
    parts = plex_id.split("_")
    # e.g. "01CPTAC_CCRCC_Proteome_JHU_20171007" → "CCRCC JHU"
    if len(parts) >= 4:
        return f"{parts[1]}\n{parts[3]}"
    return plex_id[:20]

fig, axes = plt.subplots(2, n, figsize=(4.5 * n, 9), sharey=False)
if n == 1:
    axes = axes.reshape(2, 1)

for col, plex_id in enumerate(plex_list_ordered):
    ratios = all_ratios[plex_id]

    r_unw = ratios["ratio"].replace(0, np.nan).dropna()
    r_best = (ratios
        .sort_values("precursor_intensity", ascending=False, na_position="last")
        .drop_duplicates(subset=["Peptide"], keep="first")
        ["ratio"].replace(0, np.nan).dropna())

    for row_idx, (data, color, row_title) in enumerate([
        (r_unw,  "#4878d0", "All PSMs — Unweighted"),
        (r_best, "#9467bd", "Best charge state per peptide"),
    ]):
        ax = axes[row_idx, col]
        if len(data) == 0:
            ax.text(0.5, 0.5, "No data", ha="center", va="center", transform=ax.transAxes)
        else:
            b = np.logspace(np.log10(max(data.min(), 1e-3)), np.log10(data.max()), 50)
            ax.hist(data, bins=b, color=color, edgecolor="white", linewidth=0.4)
            med = data.median()
            ax.axvline(x=1,   color="grey",    linestyle="--", linewidth=1.2)
            ax.axvline(x=med, color="#e74c3c", linestyle="-",  linewidth=1.5,
                       label=f"med = {med:.2f}")
            ax.set_xscale("log")
            ax.legend(fontsize=8)

        ax.set_xlabel("have RI / not-have RI  [log scale]")
        ax.set_ylabel("N PSMs")
        header = f"{short_label(plex_id)}\n" if row_idx == 0 else ""
        ax.set_title(f"{header}{row_title}\n(n = {len(data):,})", fontsize=9)

fig.suptitle("Per-plex channel enrichment ratio", fontsize=12, y=1.01)
plt.tight_layout()
fig.savefig(os.path.join(OUT_DIR, "plex_histograms.pdf"), bbox_inches="tight")
plt.show()

In [ ]:
# ── PLOT 2: Cross-run summary box plot ────────────────────────────────────────
# Each plex = one box.  x-axis: plex (short label).  y-axis: ratio (log).
# Shows run-to-run consistency of enrichment.

fig, axes = plt.subplots(1, 2, figsize=(max(10, 2.5 * n), 5))

for ax, col, color, title_suffix in [
    (axes[0], "ratio",     "#4878d0", "All PSMs — Unweighted"),
    (axes[1], "ratio",     "#9467bd", "Best charge state per peptide"),
]:
    data_by_plex, tick_labels = [], []
    for plex_id in plex_list_ordered:
        ratios = all_ratios[plex_id]
        if col == "ratio":
            if title_suffix.startswith("Best"):
                r = (ratios
                     .sort_values("precursor_intensity", ascending=False, na_position="last")
                     .drop_duplicates(subset=["Peptide"], keep="first")
                     ["ratio"].replace(0, np.nan).dropna())
            else:
                r = ratios["ratio"].replace(0, np.nan).dropna()
        data_by_plex.append(r.values)
        tick_labels.append(
            f"{short_label(plex_id)}\nn={len(r):,}\nmed={r.median():.2f}")

    ax.boxplot(
        data_by_plex,
        patch_artist=True,
        showfliers=True,
        flierprops=dict(marker=".", markersize=2, alpha=0.15, color=color),
        medianprops=dict(color="#e74c3c", linewidth=2),
        boxprops=dict(facecolor=color, alpha=0.45),
        whiskerprops=dict(color="grey"),
        capprops=dict(color="grey"),
    )
    ax.axhline(y=1, color="grey",      linestyle="--", linewidth=1.2, label="ratio = 1")
    ax.axhline(y=2, color="lightgrey", linestyle=":",  linewidth=1.0, label="ratio = 2")
    ax.set_yscale("log")
    ax.set_xticks(range(1, n + 1))
    ax.set_xticklabels(tick_labels, fontsize=8)
    ax.set_ylabel("mean RI (have) / mean RI (not-have)  [log scale]")
    ax.set_title(title_suffix)
    ax.legend(fontsize=8)

fig.suptitle("Cross-run enrichment ratio — runs 1–5", fontsize=11, y=1.01)
plt.tight_layout()
fig.savefig(os.path.join(OUT_DIR, "cross_run_boxplot.pdf"), bbox_inches="tight")
plt.show()

# ── Summary statistics table ───────────────────────────────────────────────────
print(f"\n{'Plex':<45} {'N PSMs':>8}  {'N uniq pep':>11}  {'Median':>7}  "
      f"{'%>1':>6}  {'%>2':>6}  {'%>5':>6}  {'FASTA ch':>9}")
print("-" * 105)
for plex_id in plex_list_ordered:
    ratios = all_ratios[plex_id]
    r = ratios["ratio"].replace(0, np.nan).dropna()
    n_uniq = ratios.drop_duplicates(subset=["Peptide"])["ratio"].notna().sum()
    fch = int(ratios["n_fasta_channels"].iloc[0]) if "n_fasta_channels" in ratios else "?"
    print(f"{plex_id[:44]:<45} {len(r):>8,}  {n_uniq:>11,}  {r.median():>7.2f}  "
          f"{100*(r>1).mean():>5.1f}%  {100*(r>2).mean():>5.1f}%  "
          f"{100*(r>5).mean():>5.1f}%  {str(fch):>9}")

In [ ]:
# ── PLOT 3: Enrichment ratio vs precursor intensity — one panel per plex ──────
# Each panel: ratio binned into 5 intensity quintiles (box plots).
# Flat trend → background RI noise is not the main cause of diluted ratios.

fig, axes = plt.subplots(1, n, figsize=(4.5 * n, 5), sharey=True)
if n == 1:
    axes = [axes]

for ax, plex_id in zip(axes, plex_list_ordered):
    ratios  = all_ratios[plex_id]
    r_prec  = ratios.dropna(subset=["precursor_intensity"]).copy()
    r_prec  = r_prec[r_prec["precursor_intensity"] > 0]

    if len(r_prec) < 10:
        ax.text(0.5, 0.5, "Insufficient\ndata", ha="center", va="center",
                transform=ax.transAxes)
        ax.set_title(short_label(plex_id))
        continue

    log_intens = np.log10(r_prec["precursor_intensity"])
    try:
        r_prec = r_prec.copy()
        r_prec["intens_bin"] = pd.qcut(log_intens, q=5, duplicates="drop")
    except ValueError:
        r_prec["intens_bin"] = pd.qcut(log_intens, q=4, duplicates="drop")

    bin_order = list(r_prec["intens_bin"].cat.categories)
    data_by_bin, tick_labels = [], []
    for b in bin_order:
        grp = r_prec[r_prec["intens_bin"] == b]["ratio"].replace(0, np.nan).dropna()
        data_by_bin.append(grp.values if len(grp) else np.array([np.nan]))
        tick_labels.append(f"10^{b.left:.1f}–\n10^{b.right:.1f}\n(n={len(grp):,})")

    ax.boxplot(
        data_by_bin,
        patch_artist=True,
        showfliers=True,
        flierprops=dict(marker=".", markersize=2, alpha=0.2, color="#4878d0"),
        medianprops=dict(color="#e74c3c", linewidth=2),
        boxprops=dict(facecolor="#4878d0", alpha=0.45),
        whiskerprops=dict(color="grey"),
        capprops=dict(color="grey"),
    )
    ax.axhline(y=1, color="grey", linestyle="--", linewidth=1.2)
    ax.set_yscale("log")
    ax.set_xticks(range(1, len(bin_order) + 1))
    ax.set_xticklabels(tick_labels, fontsize=7.5)
    ax.set_xlabel("Precursor intensity — low → high", fontsize=9)
    ax.set_title(f"{short_label(plex_id)}\n(n = {len(r_prec):,} PSMs)", fontsize=9)

axes[0].set_ylabel("mean RI (have) / mean RI (not-have)  [log scale]")

fig.suptitle(
    "Enrichment ratio vs precursor intensity — per plex\n"
    "(flat trend → background noise not driving diluted ratios)",
    fontsize=11, y=1.03)
plt.tight_layout()
fig.savefig(os.path.join(OUT_DIR, "intensity_trend_per_plex.pdf"), bbox_inches="tight")
plt.show()